In [ ]:


import warnings
warnings.filterwarnings("ignore")
from numpy import asarray
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import glob
import os
import random
import subprocess
import matplotlib.pyplot as plt
from skimage.io import imread
from matplotlib.patches import Rectangle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, Input, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")

print("Path to dataset files:", path)

100%|██████████| 2.70G/2.70G [00:18<00:00, 153MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/vipoooool/new-plant-diseases-dataset/versions/2


In [ ]:
train_dir = path + "/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train"
valid_dir = path + "/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid"
test_dir = path+"/test/test"
Diseases_classes = os.listdir(train_dir)
val_Diseases_classes = os.listdir(valid_dir)
test_Diseases_classes = os.listdir(test_dir)

In [ ]:
plt.figure(figsize=(60,60), dpi=200)
cnt = 0
plant_names = []
tot_images = 0

for i in Diseases_classes:
    cnt += 1
    plant_names.append(i)
    plt.subplot(7,7,cnt)

    images_path = os.listdir(train_dir + "/" + i)
    print("The Number of Images in " +i+ ":", len(images_path), end= " ")
    tot_images += len(images_path)

    img_show = plt.imread(train_dir + "/" + i + "/" + images_path[0])

    plt.imshow(img_show)
    plt.xlabel(i,fontsize=30)
    plt.xticks([])
    plt.yticks([])


print("\nTotal Number of Images in Directory: ", tot_images)

The Number of Images in Pepper,_bell___Bacterial_spot: 1913 The Number of Images in Apple___Apple_scab: 2016 The Number of Images in Raspberry___healthy: 1781 The Number of Images in Tomato___Tomato_Yellow_Leaf_Curl_Virus: 1961 The Number of Images in Peach___healthy: 1728 The Number of Images in Tomato___healthy: 1926 The Number of Images in Cherry_(including_sour)___Powdery_mildew: 1683 The Number of Images in Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 1642 The Number of Images in Peach___Bacterial_spot: 1838 The Number of Images in Tomato___Early_blight: 1920 The Number of Images in Potato___healthy: 1824 The Number of Images in Corn_(maize)___healthy: 1859 The Number of Images in Orange___Haunglongbing_(Citrus_greening): 2010 The Number of Images in Cherry_(including_sour)___healthy: 1826 The Number of Images in Corn_(maize)___Northern_Leaf_Blight: 1908 The Number of Images in Apple___Black_rot: 1987 The Number of Images in Blueberry___healthy: 1816 The Number of Images in

KeyboardInterrupt: 

In [ ]:
train_df=pd.DataFrame({'image':[],"label":[]})
for label in Diseases_classes:
  images_path = os.listdir(train_dir + "/" + label)
  for image in images_path:
    train_df.loc[len(train_df)] = {'image':train_dir+'/'+label+'/'+image,'label':label}
  print(label)



Apple___healthy
Blueberry___healthy
Soybean___healthy
Corn_(maize)___Common_rust_
Tomato___Tomato_mosaic_virus
Tomato___Septoria_leaf_spot
Corn_(maize)___Northern_Leaf_Blight
Tomato___Leaf_Mold
Tomato___Tomato_Yellow_Leaf_Curl_Virus
Pepper,_bell___healthy
Tomato___Late_blight
Tomato___Target_Spot
Tomato___Spider_mites Two-spotted_spider_mite
Pepper,_bell___Bacterial_spot
Tomato___Early_blight
Corn_(maize)___healthy
Potato___Late_blight
Strawberry___healthy
Apple___Cedar_apple_rust
Grape___healthy
Grape___Esca_(Black_Measles)
Peach___Bacterial_spot
Orange___Haunglongbing_(Citrus_greening)
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
Squash___Powdery_mildew
Raspberry___healthy
Cherry_(including_sour)___healthy
Tomato___Bacterial_spot
Grape___Black_rot
Strawberry___Leaf_scorch
Tomato___healthy
Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
Apple___Apple_scab
Cherry_(including_sour)___Powdery_mildew
Potato___Early_blight
Peach___healthy
Apple___Black_rot
Potato___healthy


In [ ]:
train_df

,image,label
0,/root/.cache/kagglehub/datasets/vipoooool/new-...,Apple___healthy
1,/root/.cache/kagglehub/datasets/vipoooool/new-...,Apple___healthy
2,/root/.cache/kagglehub/datasets/vipoooool/new-...,Apple___healthy
3,/root/.cache/kagglehub/datasets/vipoooool/new-...,Apple___healthy
4,/root/.cache/kagglehub/datasets/vipoooool/new-...,Apple___healthy
...,...,...
70290,/root/.cache/kagglehub/datasets/vipoooool/new-...,Potato___healthy
70291,/root/.cache/kagglehub/datasets/vipoooool/new-...,Potato___healthy
70292,/root/.cache/kagglehub/datasets/vipoooool/new-...,Potato___healthy
70293,/root/.cache/kagglehub/datasets/vipoooool/new-...,Potato___healthy


In [ ]:
val_df=pd.DataFrame({'image':[],"label":[]})
for label in val_Diseases_classes:
  images_path = os.listdir(valid_dir + "/" + label)
  for image in images_path:
    val_df.loc[len(val_df)] = {'image':valid_dir+'/'+label+'/'+image,'label':label}
  print(label)

Apple___healthy
Blueberry___healthy
Soybean___healthy
Corn_(maize)___Common_rust_
Tomato___Tomato_mosaic_virus
Tomato___Septoria_leaf_spot
Corn_(maize)___Northern_Leaf_Blight
Tomato___Leaf_Mold
Tomato___Tomato_Yellow_Leaf_Curl_Virus
Pepper,_bell___healthy
Tomato___Late_blight
Tomato___Target_Spot
Tomato___Spider_mites Two-spotted_spider_mite
Pepper,_bell___Bacterial_spot
Tomato___Early_blight
Corn_(maize)___healthy
Potato___Late_blight
Strawberry___healthy
Apple___Cedar_apple_rust
Grape___healthy
Grape___Esca_(Black_Measles)
Peach___Bacterial_spot
Orange___Haunglongbing_(Citrus_greening)
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
Squash___Powdery_mildew
Raspberry___healthy
Cherry_(including_sour)___healthy
Tomato___Bacterial_spot
Grape___Black_rot
Strawberry___Leaf_scorch
Tomato___healthy
Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
Apple___Apple_scab
Cherry_(including_sour)___Powdery_mildew
Potato___Early_blight
Peach___healthy
Apple___Black_rot
Potato___healthy


In [ ]:
val_df

,image,label
0,/root/.cache/kagglehub/datasets/vipoooool/new-...,Apple___healthy
1,/root/.cache/kagglehub/datasets/vipoooool/new-...,Apple___healthy
2,/root/.cache/kagglehub/datasets/vipoooool/new-...,Apple___healthy
3,/root/.cache/kagglehub/datasets/vipoooool/new-...,Apple___healthy
4,/root/.cache/kagglehub/datasets/vipoooool/new-...,Apple___healthy
...,...,...
17567,/root/.cache/kagglehub/datasets/vipoooool/new-...,Potato___healthy
17568,/root/.cache/kagglehub/datasets/vipoooool/new-...,Potato___healthy
17569,/root/.cache/kagglehub/datasets/vipoooool/new-...,Potato___healthy
17570,/root/.cache/kagglehub/datasets/vipoooool/new-...,Potato___healthy


In [ ]:
val_df.duplicated().sum()

0

In [ ]:
train_df.duplicated().sum()


0

In [ ]:
# def convert_image_format(image,direction):
#  return direction+'/'+image
# train_df['image']=train_df['image'].apply(lambda img : convert_image_format(img,train_dir))

In [ ]:
# val_df['image']=val_df['image'].apply(lambda img : convert_image_format(img,valid_dir))


In [ ]:

trainGenerator = ImageDataGenerator(rescale=1./255.)
valGenerator = ImageDataGenerator(rescale=1./255.)

In [ ]:
Batch_size = 32
Img_height = 224
Img_width = 224

In [ ]:
random_number=random.randint(0,len(train_df))
random_img_height = train_df['image'][random_number]
image = cv2.imread(random_img_height)
height, width= image.shape[:2]

print("The height is ", height)

print("The width is ", width)

The height is  256
The width is  256


In [ ]:
trainDataset = trainGenerator.flow_from_dataframe(
  dataframe=train_df,
  class_mode="categorical",
  x_col="image",
  y_col="label",
  batch_size=Batch_size,
  seed=42,
  shuffle=True,
  target_size=(Img_height,Img_width) #set the height and width of the images
)

valDataset = trainGenerator.flow_from_dataframe(
  dataframe=val_df,
  class_mode="categorical",
  x_col="image",
  y_col="label",
  batch_size=Batch_size,
  seed=42,
  shuffle=True,
  target_size=(Img_height,Img_width) #set the height and width of the images
)

Found 70295 validated image filenames belonging to 38 classes.
Found 17572 validated image filenames belonging to 38 classes.


In [ ]:

# Model Definition
model = models.Sequential()

model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(layers.MaxPooling2D(2, 2))

model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D(2, 2))


model.add(layers.Flatten())
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dense(38, activation='softmax'))


# model summary
model.summary()

# Compile the Model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Training the Model
history = model.fit(
    trainDataset,
    steps_per_epoch=trainDataset.samples // Batch_size,  # Number of steps per epoch
    epochs=5,  # Number of epochs
    validation_data=valDataset,
    validation_steps=valDataset.samples // Batch_size  # Validation steps
)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 222, 222, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 111, 111, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 109, 109, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 54, 54, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 186624)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │      47,776,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 38)                  │           9,766 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 47,805,158 (182.36 MB)

 Trainable params: 47,805,158 (182.36 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
2196/2196 ━━━━━━━━━━━━━━━━━━━━ 171s 75ms/step - accuracy: 0.5898 - loss: 1.6370 - val_accuracy: 0.8679 - val_loss: 0.4151
Epoch 2/5
2196/2196 ━━━━━━━━━━━━━━━━━━━━ 39s 18ms/step - accuracy: 0.8438 - loss: 0.3511 - val_accuracy: 0.8585 - val_loss: 0.4460
Epoch 3/5
2196/2196 ━━━━━━━━━━━━━━━━━━━━ 262s 95ms/step - accuracy: 0.9169 - loss: 0.2560 - val_accuracy: 0.8961 - val_loss: 0.3315
Epoch 4/5
2196/2196 ━━━━━━━━━━━━━━━━━━━━ 24s 11ms/step - accuracy: 0.9375 - loss: 0.1711 - val_accuracy: 0.8941 - val_loss: 0.3343
Epoch 5/5
2196/2196 ━━━━━━━━━━━━━━━━━━━━ 181s 69ms/step - accuracy: 0.9655 - loss: 0.1067 - val_accuracy: 0.8509 - val_loss: 0.5596


In [ ]:
# prompt: LeNet structure

def lenet_model(input_shape, num_classes):
    model = keras.Sequential(
        [
            keras.Input(shape=input_shape),
            layers.Conv2D(6, kernel_size=(5, 5), activation='relu', padding='same'),
            layers.BatchNormalization(),
            layers.MaxPooling2D(pool_size=(2, 2)),
            layers.Conv2D(16, kernel_size=(5, 5), activation='relu', padding='valid'),
            layers.BatchNormalization(),
            layers.MaxPooling2D(pool_size=(2, 2)),
            layers.Flatten(),
            layers.Dense(120, activation='relu'),
            layers.Dense(84, activation='relu'),
            layers.Dense(num_classes, activation='softmax')
        ]
    )
    return model

# Example usage:
input_shape = (Img_height, Img_width, 3)  # Assuming color images
num_classes = 38  # Number of classes in your dataset
lenet = lenet_model(input_shape, num_classes)
lenet.summary()

lenet.compile(loss=BinaryCrossentropy(),
              optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

lenet_model = lenet.fit(trainDataset, epochs=10, validation_data=valDataset)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 240, 240, 6)         │             456 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 240, 240, 6)         │              24 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 120, 120, 6)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 116, 116, 16)        │           2,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 116, 116, 16)        │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 58, 58, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 53824)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 120)                 │       6,459,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 84)                  │          10,164 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 38)                  │           3,230 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,475,354 (24.70 MB)

 Trainable params: 6,475,310 (24.70 MB)

 Non-trainable params: 44 (176.00 B)

Epoch 1/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 273s 477ms/step - accuracy: 0.3992 - loss: 0.1682 - val_accuracy: 0.3104 - val_loss: 0.1298
Epoch 2/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 217s 395ms/step - accuracy: 0.7936 - loss: 0.0336 - val_accuracy: 0.7980 - val_loss: 0.0333
Epoch 3/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 213s 387ms/step - accuracy: 0.8996 - loss: 0.0181 - val_accuracy: 0.8097 - val_loss: 0.0335
Epoch 4/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 213s 388ms/step - accuracy: 0.9538 - loss: 0.0101 - val_accuracy: 0.8479 - val_loss: 0.0287
Epoch 5/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 213s 388ms/step - accuracy: 0.9774 - loss: 0.0061 - val_accuracy: 0.8279 - val_loss: 0.0368
Epoch 6/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 213s 387ms/step - accuracy: 0.9853 - loss: 0.0045 - val_accuracy: 0.8061 - val_loss: 0.0455
Epoch 7/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 214s 390ms/step - accuracy: 0.9843 - loss: 0.0047 - val_accuracy: 0.7778 - val_loss: 0.0603
Epoch 8/10
550/550 ━━━━━━━━━━━━━━━━━━━━ 215s 391ms/step - accuracy: 0.9848 -

In [ ]:
def alexnet_model(input_shape, num_classes):
    model = keras.Sequential(
        [
            keras.Input(shape=input_shape),
            layers.Conv2D(96, kernel_size=(11, 11), strides=4, activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(pool_size=(3, 3), strides=2),
            layers.Conv2D(256, kernel_size=(5, 5), activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(pool_size=(3, 3), strides=2),
            layers.Conv2D(384, kernel_size=(3, 3), activation="relu"),
            layers.Conv2D(384, kernel_size=(3, 3), activation="relu"),
            layers.Conv2D(256, kernel_size=(3, 3), activation="relu"),
            layers.MaxPooling2D(pool_size=(3, 3), strides=2),
            layers.Flatten(),
            layers.Dense(4096, activation="relu"),
            layers.Dropout(0.5),
            layers.Dense(4096, activation="relu"),
            layers.Dropout(0.5),
            layers.Dense(num_classes, activation="softmax"),
        ]
    )
    return model

# Example usage:
input_shape = (Img_height, Img_width, 3)  # Assuming color images
num_classes = 38  # Number of classes in your dataset
alexnet = alexnet_model(input_shape, num_classes)
# alexnet = AlexNet()
alexnet.summary()

alexnet.compile(loss=BinaryCrossentropy(),
              optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

Alex_model = alexnet.fit(trainDataset, epochs=5, validation_data=valDataset)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_7 (Conv2D)                    │ (None, 58, 58, 96)          │          34,944 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 58, 58, 96)          │             384 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_5 (MaxPooling2D)       │ (None, 28, 28, 96)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_8 (Conv2D)                    │ (None, 24, 24, 256)         │         614,656 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 24, 24, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_6 (MaxPooling2D)       │ (None, 11, 11, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_9 (Conv2D)                    │ (None, 9, 9, 384)           │         885,120 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_10 (Conv2D)                   │ (None, 7, 7, 384)           │       1,327,488 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_11 (Conv2D)                   │ (None, 5, 5, 256)           │         884,992 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_7 (MaxPooling2D)       │ (None, 2, 2, 256)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_2 (Flatten)                  │ (None, 1024)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 4096)                │       4,198,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 4096)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 4096)                │      16,781,312 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 4096)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 38)                  │         155,686 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 24,884,006 (94.92 MB)

 Trainable params: 24,883,302 (94.92 MB)

 Non-trainable params: 704 (2.75 KB)

Epoch 1/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 244s 414ms/step - accuracy: 0.2667 - loss: 0.1107 - val_accuracy: 0.0903 - val_loss: 0.2846
Epoch 2/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 219s 397ms/step - accuracy: 0.7307 - loss: 0.0384 - val_accuracy: 0.6905 - val_loss: 0.0473
Epoch 3/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 219s 398ms/step - accuracy: 0.8298 - loss: 0.0254 - val_accuracy: 0.8243 - val_loss: 0.0265
Epoch 4/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 216s 393ms/step - accuracy: 0.8726 - loss: 0.0193 - val_accuracy: 0.7860 - val_loss: 0.0325
Epoch 5/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 217s 394ms/step - accuracy: 0.8932 - loss: 0.0166 - val_accuracy: 0.2395 - val_loss: 0.2033


In [ ]:
# 2️⃣ بارگذاری مدل EfficientNet-B3 از پیش‌آموزش‌داده‌شده
base_model = EfficientNetB3(weights="imagenet", include_top=False, input_shape=(Img_height, Img_width, 3))
num_classes=38
# عدم آموزش لایه‌های پایه‌ای (فریز کردن)
base_model.trainable = False

# 3️⃣ افزودن لایه‌های سفارشی برای دسته‌بندی بیماری‌های گیاهی
x = base_model.output
x=layers.BatchNormalization()(x)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.4)(x)  # جلوگیری از overfitting
predictions = Dense(num_classes, activation="softmax")(x)

# 4️⃣ ساخت مدل نهایی
model = Model(inputs=base_model.input, outputs=predictions)

# 5️⃣ کامپایل مدل
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

# 6️⃣ نمایش ساختار مدل
model.summary()

# 7️⃣ آموزش مدل
history = model.fit(
    trainDataset,
    epochs=7,
    validation_data=valDataset,
    steps_per_epoch=len(trainDataset),
    validation_steps=len(valDataset)
)


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4             │ (None, 240, 240, 3)    │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ rescaling_2 (Rescaling)   │ (None, 240, 240, 3)    │              0 │ input_layer_4[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ normalization_1           │ (None, 240, 240, 3)    │              7 │ rescaling_2[0][0]      │
│ (Normalization)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ rescaling_3 (Rescaling)   │ (None, 240, 240, 3)    │              0 │ normalization_1[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_conv_pad             │ (None, 241, 241, 3)    │              0 │ rescaling_3[0][0]      │
│ (ZeroPadding2D)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_conv (Conv2D)        │ (None, 120, 120, 40)   │          1,080 │ stem_conv_pad[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_bn                   │ (None, 120, 120, 40)   │            160 │ stem_conv[0][0]        │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_activation           │ (None, 120, 120, 40)   │              0 │ stem_bn[0][0]          │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_dwconv            │ (None, 120, 120, 40)   │            360 │ stem_activation[0][0]  │
│ (DepthwiseConv2D)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_bn                │ (None, 120, 120, 40)   │            160 │ block1a_dwconv[0][0]   │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_activation        │ (None, 120, 120, 40)   │              0 │ block1a_bn[0][0]       │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_se_squeeze        │ (None, 40)             │              0 │ block1a_activation[0]… │
│ (GlobalAveragePooling2D)  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_se_reshape        │ (None, 1, 1, 40)       │              0 │ block1a_se_squeeze[0]… │
│ (Reshape)                 │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_se_reduce         │ (None, 1, 1, 10)       │            410 │ block1a_se_reshape[0]… │
│ (Conv2D)                  │                        │                │                        │
├──────────────────────

 Total params: 11,192,917 (42.70 MB)

 Trainable params: 406,310 (1.55 MB)

 Non-trainable params: 10,786,607 (41.15 MB)

Epoch 1/7


KeyboardInterrupt: 

In [ ]:

# prompt: EfficientNet structure with BatchNormalization

import tensorflow as tf
from tensorflow import keras

def EfficientNetBlock(input_tensor, num_filters, expand_ratio, strides=1):
    # Expansion Phase
    expanded_filters = int(num_filters * expand_ratio)
    x = layers.Conv2D(expanded_filters, 1, padding='same', use_bias=False)(input_tensor)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x) # EfficientNet uses Swish activation

    # Depthwise Convolution
    x = layers.DepthwiseConv2D(3, strides=strides, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)

    # Squeeze and Excitation (SE) block (optional, but improves performance)
    # ... (Add SE block code here if needed)

    # Projection Phase
    x = layers.Conv2D(num_filters, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    # Residual Connection
    if strides == 1 and input_tensor.shape[-1] == num_filters:
        x = layers.add([x, input_tensor])

    return x

def EfficientNet(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    # Initial Convolution
    x = layers.Conv2D(32, 3, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)

    # EfficientNet Blocks (example configuration, adjust as needed)
    x = EfficientNetBlock(x, 16, 1)
    x = EfficientNetBlock(x, 24, 6, strides=2)
    x = EfficientNetBlock(x, 40, 6, strides=2)
    # ... Add more EfficientNet blocks

    # Classification Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(1280, activation='swish')(x) # Bottleneck layer
    x = layers.Dropout(0.2)(x)  # Dropout for regularization
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model


# Example usage:
input_shape = (Img_height, Img_width, 3)  # Assuming color images
num_classes = 38  # Number of classes in your dataset
efficientnet = EfficientNet(input_shape, num_classes)
efficientnet.summary()

efficientnet.compile(loss=BinaryCrossentropy(),
              optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])
efficientnet_model = efficientnet.fit(trainDataset, epochs=5, validation_data=valDataset)

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)           │ (None, 240, 240, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_12 (Conv2D)                   │ (None, 120, 120, 32)        │             864 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_7                │ (None, 120, 120, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation (Activation)              │ (None, 120, 120, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_13 (Conv2D)                   │ (None, 120, 120, 16)        │             512 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_8                │ (None, 120, 120, 16)        │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_1 (Activation)            │ (None, 120, 120, 16)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ depthwise_conv2d (DepthwiseConv2D)   │ (None, 120, 120, 16)        │             144 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_9                │ (None, 120, 120, 16)        │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_2 (Activation)            │ (None, 120, 120, 16)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_14 (Conv2D)                   │ (None, 120, 120, 16)        │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_10               │ (None, 120, 120, 16)        │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_15 (Conv2D)                   │ (None, 120, 120, 144)       │           2,304 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_11               │ (None, 120, 120, 144)       │             576 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_3 (Activation)            │ (None, 120, 120, 144)       │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ depthwise_conv2d_1 (DepthwiseConv2D) │ (None, 60, 60, 144)         │           1,296 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_12               │ (None, 60, 60, 144)         │             576 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_4 (Activation)            │ (None, 60, 60, 144)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

 Total params: 131,158 (512.34 KB)

 Trainable params: 129,334 (505.21 KB)

 Non-trainable params: 1,824 (7.12 KB)

Epoch 1/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 281s 468ms/step - accuracy: 0.2142 - loss: 0.1894 - val_accuracy: 0.1431 - val_loss: 0.1765
Epoch 2/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 243s 443ms/step - accuracy: 0.7425 - loss: 0.0403 - val_accuracy: 0.7620 - val_loss: 0.0386
Epoch 3/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 244s 443ms/step - accuracy: 0.8601 - loss: 0.0244 - val_accuracy: 0.7547 - val_loss: 0.0395
Epoch 4/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 244s 443ms/step - accuracy: 0.9036 - loss: 0.0179 - val_accuracy: 0.8160 - val_loss: 0.0311
Epoch 5/5
550/550 ━━━━━━━━━━━━━━━━━━━━ 246s 447ms/step - accuracy: 0.9246 - loss: 0.0146 - val_accuracy: 0.8265 - val_loss: 0.0299


In [ ]:
def ConvBlock(in_channels, out_channels, pool=False):
    layers = [nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
             nn.BatchNorm2d(out_channels),
             nn.ReLU(inplace=True)]
    if pool:
        layers.append(nn.MaxPool2d(4))
    return nn.Sequential(*layers)
    # resnet architecture
class ResNet9(ImageClassificationBase):
    def __init__(self, in_channels, num_diseases):
        super().__init__()

        self.conv1 = ConvBlock(in_channels, 64)
        self.conv2 = ConvBlock(64, 128, pool=True) # out_dim : 128 x 64 x 64
        self.res1 = nn.Sequential(ConvBlock(128, 128), ConvBlock(128, 128))

        self.conv3 = ConvBlock(128, 256, pool=True) # out_dim : 256 x 16 x 16
        self.conv4 = ConvBlock(256, 512, pool=True) # out_dim : 512 x 4 x 44
        self.res2 = nn.Sequential(ConvBlock(512, 512), ConvBlock(512, 512))

        self.classifier = nn.Sequential(nn.MaxPool2d(4),
                                       nn.Flatten(),
                                       nn.Linear(512, num_diseases))

    def forward(self, xb): # xb is the loaded batch
        out = self.conv1(xb)
        out = self.conv2(out)
        out = self.res1(out) + out
        out = self.conv3(out)
        out = self.conv4(out)
        out = self.res2(out) + out
        out = self.classifier(out)
        return out

In [ ]:
def resnet_model(input_shape, num_classes):
    model = keras.Sequential(
        [
            keras.Input(shape=input_shape),
            layers.Conv2D(64, kernel_size=(3, 3),padding='same', activation="relu"),
            layers.BatchNormalization(),
            layers.Conv2D(128, kernel_size=(3,3), activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(pool_size=(4,4), strides=4),
            layers.Conv2D(128, kernel_size=(3, 3), activation="relu"),
            layers.BatchNormalization(),
            layers.Conv2D(128, kernel_size=(3, 3), activation="relu"),
            layers.BatchNormalization(),
            layers.Conv2D(256, kernel_size=(3, 3), activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(pool_size=(4,4), strides=4),
            layers.Conv2D(512, kernel_size=(3, 3), activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(pool_size=(4,4), strides=4),
            layers.Conv2D(512, kernel_size=(3, 3), activation="relu"),
            layers.BatchNormalization(),
            layers.Conv2D(512, kernel_size=(3, 3), activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(pool_size=(4,4), strides=4),
            layers.Flatten(),
            # layers.Dense(4096, activation="relu"),
            # layers.Dropout(0.5),
            # layers.Dense(4096, activation="relu"),
            # layers.Dropout(0.5),
            layers.Dense(num_classes, activation="softmax"),
        ]
    )
    return model

# Example usage:
input_shape = (Img_height, Img_width, 3)  # Assuming color images
num_classes = 38  # Number of classes in your dataset
resnet = resnet_model(input_shape, num_classes)
# alexnet = AlexNet()
resnet.summary()

resnet.compile(loss=BinaryCrossentropy(),
              optimizer=Adam(learning_rate=0.01), metrics=['accuracy'])

Resnet_model = resnet.fit(trainDataset, epochs=5, validation_data=valDataset)

ValueError: Computed output size would be negative. Received `inputs shape=(None, 0, 0, 512)`, `kernel shape=(3, 3, 512, 512)`, `dilation_rate=[1 1]`.

In [ ]:
def residual_block(filters):
    def block(x):
        shortcut = x
        x = layers.Conv2D(filters, (3,3), padding='same', activation=None)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Conv2D(filters, (3,3), padding='same', activation=None)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Add()([x, shortcut])
        return x
    return block

def ResNet9(input_shape=(32, 32, 3), num_classes=38):
    inputs = layers.Input(shape=input_shape)

    # Conv1
    x = layers.Conv2D(64, (3,3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Conv2 + MaxPool
    x = layers.Conv2D(128, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((4,4))(x)

    # Residual Block 1
    x = residual_block(128)(x)

    # Conv3 + MaxPool
    x = layers.Conv2D(256, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((4,4))(x)

    # Conv4 + MaxPool
    x = layers.Conv2D(512, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((4,4))(x)

    # Residual Block 2
    x = residual_block(512)(x)

    # Classifier
    x = layers.MaxPooling2D((4,4))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, x)
    return model
input_shape = (Img_height, Img_width, 3)  # Assuming color images
num_classes = 38  # Number of classes in your dataset
# مدل را بسازید
model = ResNet9(input_shape,num_classes)
model.summary()

model.compile(loss=BinaryCrossentropy(),
              optimizer=Adam(learning_rate=0.01), metrics=['accuracy'])

Resnet_model = model.fit(trainDataset, epochs=5, validation_data=valDataset)

Model: "functional_30"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_14            │ (None, 224, 224, 3)    │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_85 (Conv2D)        │ (None, 224, 224, 64)   │          1,792 │ input_layer_14[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_88    │ (None, 224, 224, 64)   │            256 │ conv2d_85[0][0]        │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_40 (ReLU)           │ (None, 224, 224, 64)   │              0 │ batch_normalization_8… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_86 (Conv2D)        │ (None, 224, 224, 128)  │         73,856 │ re_lu_40[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_89    │ (None, 224, 224, 128)  │            512 │ conv2d_86[0][0]        │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_41 (ReLU)           │ (None, 224, 224, 128)  │              0 │ batch_normalization_8… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d_43          │ (None, 56, 56, 128)    │              0 │ re_lu_41[0][0]         │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_87 (Conv2D)        │ (None, 56, 56, 128)    │        147,584 │ max_pooling2d_43[0][0] │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_90    │ (None, 56, 56, 128)    │            512 │ conv2d_87[0][0]        │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_42 (ReLU)           │ (None, 56, 56, 128)    │              0 │ batch_normalization_9… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_88 (Conv2D)        │ (None, 56, 56, 128)    │        147,584 │ re_lu_42[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_91    │ (None, 56, 56, 128)    │            512 │ conv2d_88[0][0]        │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ re_lu_43 (ReLU)           │ (None, 56, 56, 128)    │              0 │ batch_normalization_9… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_10 (Add)              │ (None, 56, 56, 128)    │              0 │ re_lu_43[0][0],        │
│                           │                        │                │ max_pooling2d_43[0][0] │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_89 (Conv2D)        │ (None, 56, 56, 256)    │        295,168 │ add_10[0][0]           │
├──────────────────────

 Total params: 6,574,758 (25.08 MB)

 Trainable params: 6,570,278 (25.06 MB)

 Non-trainable params: 4,480 (17.50 KB)

Epoch 1/5


ValueError: Exception encountered when calling MaxPooling2D.call().

[1mNegative dimension size caused by subtracting 4 from 3 for '{{node functional_30_1/max_pooling2d_46_1/MaxPool2d}} = MaxPool[T=DT_FLOAT, data_format="NHWC", explicit_paddings=[], ksize=[1, 4, 4, 1], padding="VALID", strides=[1, 4, 4, 1]](functional_30_1/add_11_1/Add)' with input shapes: [?,3,3,512].[0m

Arguments received by MaxPooling2D.call():
  • inputs=tf.Tensor(shape=(None, 3, 3, 512), dtype=float32)

NameError: name 'resnet' is not defined

In [ ]:
import tensorflow
from tensorflow import keras
from keras.models import Sequential,load_model,Model
from keras.layers import Conv2D,MaxPool2D,AveragePooling2D,Dense,Flatten,ZeroPadding2D,BatchNormalization,Activation,Add,Input,Dropout,GlobalAveragePooling2D
from keras.optimizers import SGD
from keras.initializers import glorot_uniform
# from keras.preprocessing import ImageDataGenerator
from keras.callbacks import ModelCheckpoint,EarlyStopping,ReduceLROnPlateau
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

In [ ]:
base_model_tf=ResNet50(include_top=False,weights='imagenet',input_shape=(224,224,3),classes=38)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [ ]:
#Model building
base_model_tf.trainable=False

pt=Input(shape=(224,224,3))
# func=tensorflow.cast(pt,tensorflow.float32)
# x=preprocess_input(func) #This function used to zero-center each color channel wrt Imagenet dataset
model_resnet=base_model_tf(pt,training=False)
model_resnet=GlobalAveragePooling2D()(model_resnet)
model_resnet=Dense(128,activation='relu')(model_resnet)
model_resnet=Dense(64,activation='relu')(model_resnet)
model_resnet=Dense(38,activation='softmax')(model_resnet)


model_main=Model(inputs=pt,outputs=model_resnet)
model_main.summary()

Model: "functional_31"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_15 (InputLayer)          │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ resnet50 (Functional)                │ (None, 7, 7, 2048)          │      23,587,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_1           │ (None, 2048)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_24 (Dense)                     │ (None, 128)                 │         262,272 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_25 (Dense)                     │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_26 (Dense)                     │ (None, 38)                  │           2,470 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 23,860,710 (91.02 MB)

 Trainable params: 272,998 (1.04 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
#CallBacks
es=EarlyStopping(monitor='val_accuracy',verbose=1,patience=7,mode='auto')
mc=ModelCheckpoint(filepath='/content.keras',monitor='val_accuracy',verbose=1,save_best_only=True)
lr=ReduceLROnPlateau(monitor='val_accuracy',verbose=1,patience=5,min_lr=0.001)

In [ ]:
model_main.compile(optimizer='Adam',loss='categorical_crossentropy',metrics=['accuracy'])


In [ ]:
model_main.fit(trainDataset,validation_data=valDataset,epochs=6,steps_per_epoch=200,verbose=1,callbacks=[mc,es,lr])


Epoch 1/6
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 401ms/step - accuracy: 0.0441 - loss: 3.5896
Epoch 1: val_accuracy improved from -inf to 0.09350, saving model to /content.keras
200/200 ━━━━━━━━━━━━━━━━━━━━ 166s 828ms/step - accuracy: 0.0442 - loss: 3.5892 - val_accuracy: 0.0935 - val_loss: 3.3011 - learning_rate: 0.0010
Epoch 2/6
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 386ms/step - accuracy: 0.0993 - loss: 3.2608
Epoch 2: val_accuracy improved from 0.09350 to 0.13442, saving model to /content.keras
200/200 ━━━━━━━━━━━━━━━━━━━━ 125s 626ms/step - accuracy: 0.0993 - loss: 3.2605 - val_accuracy: 0.1344 - val_loss: 3.0651 - learning_rate: 0.0010
Epoch 3/6
150/200 ━━━━━━━━━━━━━━━━━━━━ 18s 379ms/step - accuracy: 0.1541 - loss: 3.0142
Epoch 3: val_accuracy improved from 0.13442 to 0.18649, saving model to /content.keras
200/200 ━━━━━━━━━━━━━━━━━━━━ 105s 524ms/step - accuracy: 0.1567 - loss: 3.0037 - val_accuracy: 0.1865 - val_loss: 2.8679 - learning_rate: 0.0010
Epoch 4/6
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 362